# Standardize output data
Run these cells sequentially

In [4]:
# Import necessary modules
import os
from utils import *
import utils.for_setpath as path

# Run analysis

#### Within-subject

In [5]:
results_dir = os.path.join(results, 'fulldata')
design      = 'within_subject'
sub         = 'OE012'
ses         = 'Norreport'
sub_data    = do_analysis_design(results_dir, design, subject=sub, session=ses)

No data found for sub-OE012, ses-Norreport.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import contextily as ctx
import matplotlib.ticker as mticker
from shapely.geometry import Point

def plot_z_normalized_map_geometry(df, variable, title="Z-Normalized Map", buffer_ratio=0.1, 
                                 rolling_window=None, window_type='gaussian', min_periods=1):
    """
    Plot z-normalized values of a variable on a map using latitude and longitude coordinates,
    with optional rolling mean smoothing and symmetric colorbar.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing latitude_corrected, longitude_corrected and the variable to plot
    variable : str
        Name of the variable to plot
    title : str, optional
        Title for the plot (default: "Z-Normalized Map")
    buffer_ratio : float, optional
        Ratio to extend the map bounds beyond the data points (default: 0.1 = 10%)
    rolling_window : int, optional
        Size of the rolling window for smoothing. If None, no smoothing is applied.
    window_type : str, optional
        Type of window for rolling mean ('gaussian', 'boxcar', etc.). Default is 'gaussian'.
    min_periods : int, optional
        Minimum number of observations required for rolling mean calculation.
    """
    # 1) Check if needed columns exist
    required_cols = ['latitude_corrected', 'longitude_corrected', variable]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"Missing required columns: {', '.join(missing_cols)}")
        return

    # 2) Create a copy and drop rows with missing coordinates or variable
    df = df.copy()
    df = df.dropna(subset=['latitude_corrected', 'longitude_corrected', variable])
    
    if df.empty:
        print("No valid rows after dropping NaNs.")
        return

    # 3) Apply rolling mean if specified
    if rolling_window is not None:
        # Sort by time if available, otherwise by index
        if 'timestamp' in df.columns:
            df = df.sort_values('timestamp')
        
        # Apply rolling mean
        if window_type == 'gaussian':
            # For gaussian window, std = window_size/5 is a good rule of thumb
            std = rolling_window/5
            df[variable] = df[variable].rolling(
                window=rolling_window,
                win_type='gaussian',
                min_periods=min_periods,
                center=True
            ).mean(std=std)
        else:
            df[variable] = df[variable].rolling(
                window=rolling_window,
                win_type=window_type,
                min_periods=min_periods,
                center=True
            ).mean()
        
        # Drop NaN values that might have been introduced
        df = df.dropna(subset=[variable])

    # 4) Create Point geometries from coordinates
    try:
        geometry = [Point(xy) for xy in zip(df['longitude_corrected'], df['latitude_corrected'])]
        gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
    except Exception as e:
        print(f"Error creating GeoDataFrame: {e}")
        return

    # 5) Compute z-score for the chosen variable
    mean_val = gdf[variable].mean()
    std_val = gdf[variable].std()
    if std_val == 0 or np.isnan(std_val):
        print("Std is zero/NaN, plotting raw values for variable.")
        gdf["z_score"] = gdf[variable]
    else:
        gdf["z_score"] = (gdf[variable] - mean_val) / std_val

    # 6) Reject outliers with abs(z-score) > 3
    gdf = gdf.loc[gdf["z_score"].abs() <= 3].copy()
    if gdf.empty:
        print("All points were outliers (|z|>3). Nothing to plot.")
        return

    # 7) Convert to Web Mercator for plotting
    gdf_web = gdf.to_crs(epsg=3857)
    
    # Calculate bounds in Web Mercator
    bounds = gdf_web.total_bounds
    x_range = bounds[2] - bounds[0]
    y_range = bounds[3] - bounds[1]
    x_buffer = x_range * buffer_ratio
    y_buffer = y_range * buffer_ratio
    
    # Create figure and axis
    fig, ax = plt.subplots(figsize=(12, 8))

    # Add basemap first
    ax.axis('equal')
    bounds_web = [
        bounds[0] - x_buffer,
        bounds[2] + x_buffer,
        bounds[1] - y_buffer,
        bounds[3] + y_buffer
    ]
    
    # Set limits in Web Mercator
    ax.set_xlim(bounds_web[0], bounds_web[1])
    ax.set_ylim(bounds_web[2], bounds_web[3])

    try:
        ctx.add_basemap(
            ax,
            source=ctx.providers.CartoDB.Positron,
            zoom='auto'
        )
    except Exception as e:
        print(f"Could not add basemap with contextily: {e}")

    # Find symmetric vmin/vmax for colorbar
    max_abs_z = max(abs(gdf_web["z_score"].min()), abs(gdf_web["z_score"].max()))
    
    # Plot points with symmetric colorbar
    scatter = gdf_web.plot(
        ax=ax,
        column="z_score",
        cmap="coolwarm",
        alpha=0.8,
        legend=True,
        markersize=100,
        zorder=5,
        vmin=-max_abs_z,
        vmax=max_abs_z
    )

    # Convert Web Mercator coordinates to lat/lon for display
    @mticker.FuncFormatter
    def mercator_to_lon(x, p):
        return f"{x/20037508.34*180:.2f}°"

    @mticker.FuncFormatter
    def mercator_to_lat(y, p):
        lat = np.arcsinh(np.exp(y/20037508.34*np.pi))*180/np.pi - 90
        return f"{lat:.2f}°"

    # Set coordinate formatters
    ax.xaxis.set_major_formatter(mercator_to_lon)
    ax.yaxis.set_major_formatter(mercator_to_lat)

    # Customize the plot
    ax.set_title(title, fontsize=14, pad=20)
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    return gdf

result = plot_z_normalized_map_geometry(
    sub_data, 
    variable="frontal alpha", 
    title="Z-Normalized Alpha Map (Smoothed)",
    rolling_window=15,  # Adjust window size as needed
    window_type='gaussian'
)


#### Betweeen-subject

In [6]:
results_dir = os.path.join(results, 'fulldata')
design = 'between_subject'
between_subject_data = do_analysis_design(results_dir, design)
# Export
results_dir = os.path.join(results, 'fulldata')
output_dir  = os.path.join(results, 'fulldata_analysis',design)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
between_subject_data.to_csv(os.path.join(output_dir, 'between_subject.csv'), index=False)

c:\github\JoaoAmaro2001\notebooks\src\ingestion\utils\for_analysis.py:107: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(dataframes, ignore_index=True)
c:\github\JoaoAmaro2001\notebooks\src\ingestion\utils\for_analysis.py:107: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(dataframes, ignore_index=True)
c:\github\JoaoAmaro2001\notebooks\src\ingestion\utils\for_analysis.py:107: FutureWarning: The behavior of DataFrame concatenation with empty or all-

##### Full covariance and correlation matrices

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


def compute_matrices(df, variables):
    """
    Compute the correlation and covariance matrices for a given set of variables.
    Args:
        df (pd.DataFrame): DataFrame containing the variables.
        variables (list): List of variable names to include in the matrices.

    Returns:
        corr_matrix (pd.DataFrame): Correlation matrix.
        cov_matrix (pd.DataFrame): Covariance matrix.
    """
    available_vars = [col for col in variables if col in df.columns]
    if not available_vars:
        raise ValueError("None of the specified variables are in the DataFrame.")

    # Correlation matrix
    corr_matrix = df[available_vars].corr()

    # Covariance matrix
    cov_matrix = df[available_vars].cov()

    return corr_matrix, cov_matrix


def plot_square_matrix(matrix, title, output_path, vmin=None, vmax=None, cmap="coolwarm", fmt=".2f"):
    """
    Plot a square matrix (e.g., correlation or covariance) as a heatmap.
    Args:
        matrix (pd.DataFrame): The matrix to plot.
        title (str): Title of the plot.
        output_path (str): Path to save the plot.
        vmin (float): Minimum value for the colormap.
        vmax (float): Maximum value for the colormap.
        cmap (str): Colormap to use.
        fmt (str): Format for annotations.

    Returns:
        None: Saves the plot to the specified path.
    """
    plt.figure(figsize=(10, 10))
    ax = sns.heatmap(
        matrix,
        annot=False,
        fmt=fmt,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        square=True,
        cbar_kws={"shrink": 0.8, "label": "Value"},
        linewidths=0.5,
        annot_kws={"fontsize": 8},
    )
    ax.set_title(title, fontsize=16, weight="bold")
    ax.tick_params(axis="x", labelsize=10, rotation=-90)
    ax.tick_params(axis="y", labelsize=10)
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=10)
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


# Define sets of variables
EEG_VARS = [
    "delta", "theta", "alpha", "beta", "gamma", "frontal alpha",
    "frontal midline theta", "theta-beta ratio", "frontal alpha asymmetry", "frontal theta"
]

PHYSIO_VARS = [
    "empatica_e4_ibi", "skin_temperature", "eda_raw", "eda_phasic",
    "x_acceleration", "y_acceleration", "z_acceleration", "acceleration_magnitude",
    "bvp", "empatica_heart_rate"
]

CLIMATE_VARS = [
    "air_temperature", "air_humidity", "air_pressure", "sound_pressure_level",
    "pm2_5", "pm10_0", "solar_light", "thermocouple_temp",
    "north_wind", "east_wind", "gust_wind", "wind_speed", "radiant_temp", "utci"
]

# Combine all variables
ALL_VARS = EEG_VARS + PHYSIO_VARS + CLIMATE_VARS

# Load data
output_dir = os.path.join(results, "fulldata_analysis", "between_subject")
data_path = os.path.join(output_dir, "between_subject.csv")
df = pd.read_csv(data_path)

# Compute correlation and covariance matrices
corr_matrix, cov_matrix = compute_matrices(df, ALL_VARS)

# Save matrices as CSV
corr_matrix.to_csv(os.path.join(output_dir, "all_vars_corr_matrix.csv"))
cov_matrix.to_csv(os.path.join(output_dir, "all_vars_cov_matrix.csv"))

# Plot correlation matrix
plot_square_matrix(
    corr_matrix,
    title="Correlation Matrix of All Variables",
    output_path=os.path.join(output_dir, "correlation_matrix.png"),
    vmin=-1,
    vmax=1,
    cmap="coolwarm",
)

# Plot covariance matrix
plot_square_matrix(
    cov_matrix,
    title="Covariance Matrix of All Variables",
    output_path=os.path.join(output_dir, "covariance_matrix.png"),
    cmap="viridis",
    fmt=".1f",
)

print("Done: Matrices computed, exported, and plots saved.")


##### Covariance matrices masked for p- and r-values

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt

def compute_physio_climate_correlations(df, phys_vars, clim_vars, alpha=0.05):
    """
    Compute Pearson correlations (r, p) between each variable in phys_vars and each in clim_vars.
    Returns:
      corr_r: DataFrame of correlation coefficients, shape=(len(phys_vars), len(clim_vars))
      corr_p: DataFrame of p-values, same shape
      mask_significant: boolean DataFrame, True if p < alpha
    """
    phys_available = [col for col in phys_vars if col in df.columns]
    clim_available = [col for col in clim_vars if col in df.columns]

    r_values = np.zeros((len(phys_available), len(clim_available)), dtype=float)
    p_values = np.ones((len(phys_available), len(clim_available)), dtype=float)

    for i, pcol in enumerate(phys_available):
        for j, ccol in enumerate(clim_available):
            subdata = df[[pcol, ccol]].dropna()
            if len(subdata) > 2:
                r, p = pearsonr(subdata[pcol], subdata[ccol])
                r_values[i, j] = r
                p_values[i, j] = p
            else:
                r_values[i, j] = np.nan
                p_values[i, j] = np.nan

    corr_r = pd.DataFrame(r_values, index=phys_available, columns=clim_available)
    corr_p = pd.DataFrame(p_values, index=phys_available, columns=clim_available)
    mask_significant = corr_p < alpha

    return corr_r, corr_p, mask_significant


def plot_correlation_matrix(
    corr_r, corr_p, mask_sig, title, out_png, mask_abs_r=None
):
    """
    Plot correlation heatmaps:
    - Full correlation matrix
    - Masked correlation matrix for significance and optional |r| threshold.
    """
    def customize_heatmap(ax, title):
        ax.set_title(title, fontsize=16, weight="bold")
        ax.tick_params(axis="x", labelsize=12, rotation=-90)
        ax.tick_params(axis="y", labelsize=12)
        cbar = ax.collections[0].colorbar
        cbar.ax.tick_params(labelsize=12)

    # Mask for significant correlations
    if mask_abs_r is not None:
        combined_mask = mask_sig & (corr_r.abs() >= mask_abs_r)
    else:
        combined_mask = mask_sig

    # 1) Full correlation matrix
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(
        corr_r,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1,
        cbar_kws={"label": "Correlation Coefficient (r)"},
    )
    customize_heatmap(ax, f"{title}")
    plt.tight_layout()
    plt.savefig(out_png.replace(".png", "_full.png"), dpi=150)
    plt.close()

    # 2) Masked correlation matrix (significance + optional |r| threshold)
    masked_corr = corr_r.where(combined_mask, np.nan)
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(
        masked_corr,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1,
        cbar_kws={"label": "Correlation Coefficient (r)"},
    )
    customize_heatmap(
        ax,
        f"{title} (p < 0.05"
        + (f" & |r| >= {mask_abs_r}" if mask_abs_r else "") + ")",
    )
    plt.tight_layout()
    plt.savefig(out_png.replace(".png", "_masked.png"), dpi=150)
    plt.close()

EEG_VARS = [
    # EEG variables
    "delta","theta","alpha","beta","gamma","frontal alpha","frontal midline theta",
    "theta-beta ratio","frontal alpha asymmetry","frontal theta",
]

PHYSIO_VARS = [
    "empatica_e4_ibi",
    "skin_temperature","eda_raw","eda_phasic","x_acceleration","y_acceleration","z_acceleration",
    "acceleration_magnitude","bvp", "empatica_heart_rate"
    # Add or remove columns as you see fit
]

CLIMATE_VARS = [
    "air_temperature","air_humidity","air_pressure","sound_pressure_level",
    "pm2_5","pm10_0","solar_light","thermocouple_temp",
    "north_wind","east_wind","gust_wind",
    "wind_speed","radiant_temp","utci"
    # Add or remove columns as you see fit
]

# MAIN EXECUTION
output_dir = os.path.join(results, "fulldata_analysis", "between_subject")
between_subject_data = pd.read_csv(os.path.join(output_dir, "between_subject.csv"))

# Compute correlation
corr_r, corr_p, mask_sig = compute_physio_climate_correlations(
    between_subject_data, PHYSIO_VARS, CLIMATE_VARS, alpha=0.05
)

# Save correlation matrices
corr_r.to_csv(os.path.join(output_dir, "physio_climate_corr_r.csv"))
corr_p.to_csv(os.path.join(output_dir, "physio_climate_corr_p.csv"))

# Plot with a user-defined mask for |r| >= 0.5
plot_correlation_matrix(
    corr_r,
    corr_p,
    mask_sig,
    title="Correlation between Physiological and Climate Measurements",
    out_png=os.path.join(output_dir, "physio_climate_correlation.png"),
    mask_abs_r=0.2,  # Example: Mask correlations where |r| < 0.5
)

print("Done: correlation matrices exported and plots saved.")


In [ ]:
# Check differences between age groups

#### Within-path

In [ ]:
results_dir      = os.path.join(results, 'fulldata')
design           = 'within_path'
ses              = 'Maat'
data_within_path = do_analysis_design(results_dir, design, session=ses)
# Export
results_dir      = os.path.join(results, 'fulldata')
output_dir       = os.path.join(results, 'fulldata_analysis',design, ses)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
data_within_path.to_csv(os.path.join(output_dir, f'{ses}.csv'), index=False)

###### Plot averaged metrics

In [ ]:
def plot_participant_averaged_map(df, variable, participant_col, title="Participant-Averaged Map", output_path=None):
    """
    Plot averaged values across participants for each coordinate on a map.
    
    Args:
        df (pd.DataFrame): DataFrame containing 'geometry' (WKT or shapely objects), participant ID, and the variable
        variable (str): The column to plot
        participant_col (str): Column name containing participant IDs
        title (str): Title for the plot
        output_path (str, optional): Path to save the plot. If None, plot is only displayed
    
    Returns:
        None: Displays and optionally saves the plot
    """
    import pandas as pd
    import numpy as np
    import geopandas as gpd
    import shapely.wkt
    import contextily as ctx
    import matplotlib.pyplot as plt
    from shapely.geometry import Point
    import matplotlib.ticker as mticker
    from pyproj import Transformer
    import warnings
    warnings.filterwarnings('ignore', category=UserWarning)

    # Ensure geometry objects
    def ensure_geom(x):
        if isinstance(x, str):
            return shapely.wkt.loads(x)
        return x

    # Average values across participants for each coordinate
    grouped = (
        df.groupby(["longitude_corrected", "latitude_corrected"], dropna=False)[variable]
        .mean()
        .reset_index()
    )

    if grouped.empty:
        print("No valid data after grouping. Nothing to plot.")
        return

    # Create GeoDataFrame
    grouped["geometry"] = [
        Point(xy) for xy in zip(grouped["longitude_corrected"], grouped["latitude_corrected"])
    ]
    gdf = gpd.GeoDataFrame(grouped, geometry="geometry", crs="EPSG:4326")

    # Create intervals for plotting
    vmin = gdf[variable].min()
    vmax = gdf[variable].max()
    bin_edges = np.linspace(vmin, vmax, 6)  # 5 bins
    
    # Ensure bin edges are unique
    bin_edges = np.unique(bin_edges)
    if len(bin_edges) < 2:
        bin_edges = np.array([bin_edges[0] - 0.5, bin_edges[0] + 0.5])

    # Create intervals
    try:
        gdf["interval"] = pd.cut(
            gdf[variable],
            bins=bin_edges,
            include_lowest=True,
            labels=[f"{bin_edges[i]:.2f} to {bin_edges[i+1]:.2f}" 
                   for i in range(len(bin_edges)-1)]
        )
    except ValueError as e:
        print(f"Warning: Fallback to quartile binning due to: {e}")
        gdf["interval"] = pd.qcut(
            gdf[variable],
            q=4,
            duplicates='drop',
            labels=['Q1', 'Q2', 'Q3', 'Q4']
        )

    # Convert to Web Mercator
    try:
        gdf_web = gdf.to_crs(epsg=3857)
    except Exception as e:
        print(f"Could not convert to EPSG:3857 (Web Mercator): {e}")
        gdf_web = None

    # Create coordinate transformer
    transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

    def transform_x(x, _):
        """Transform Web Mercator x coordinate to longitude"""
        lon, _ = transformer.transform(x, 0)
        return f"{lon:.3f}°"

    def transform_y(y, _):
        """Transform Web Mercator y coordinate to latitude"""
        _, lat = transformer.transform(0, y)
        return f"{lat:.3f}°"

    # Plot
    fig, ax = plt.subplots(figsize=(15, 10))

    if gdf_web is not None:
        gdf_web.plot(
            ax=ax,
            column="interval",
            cmap="viridis",  # Changed from coolwarm since we're not showing z-scores
            alpha=0.8,
            legend=True,
            markersize=40,
        )
        try:
            ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
        except Exception as e:
            print(f"Could not add basemap with contextily: {e}")
        
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(transform_x))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(transform_y))
    else:
        gdf.plot(
            ax=ax,
            column="interval",
            cmap="viridis",  # Changed from coolwarm
            alpha=0.8,
            legend=True,
            markersize=40,
        )
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}°"))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.1f}°"))
        
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)
    ax.set_title(title, fontsize=14)

    # Handle plot saving
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
    
    plt.show()


# Run
plot_participant_averaged_map(
    df=data_within_path,
    variable='noise_level',
    participant_col='participant_id',
    title='Average Values Across Participants',
    output_path=os.path.join(output_dir, "Maat_raw.png")
)

###### Plot z-normalized metrics

In [ ]:
def plot_participant_averaged_zmap(df, variable, participant_col, z_threshold=3, title="Participant-Averaged Z-Score Map", output_path=None):
    """
    Plot averaged z-normalized values across participants for each coordinate on a map.
    
    Args:
        df (pd.DataFrame): DataFrame containing 'geometry' (WKT or shapely objects), participant ID, and the variable
        variable (str): The column to z-normalize and plot
        participant_col (str): Column name containing participant IDs
        z_threshold (float): Threshold for removing outliers by absolute z-score. Default is 3
        title (str): Title for the plot
        output_path (str, optional): Path to save the plot. If None, plot is only displayed
    
    Returns:
        None: Displays and optionally saves the plot
    """
    import pandas as pd
    import numpy as np
    import geopandas as gpd
    import shapely.wkt
    import contextily as ctx
    import matplotlib.pyplot as plt
    from shapely.geometry import Point
    import matplotlib.ticker as mticker
    from pyproj import Transformer
    import warnings
    warnings.filterwarnings('ignore', category=UserWarning)

    # Ensure geometry objects
    def ensure_geom(x):
        if isinstance(x, str):
            return shapely.wkt.loads(x)
        return x

    # Compute z-scores per participant
    def compute_participant_zscores(group):
        mean_val = group[variable].mean()
        std_val = group[variable].std()
        
        if std_val == 0 or np.isnan(std_val):
            # Use min-max scaling if std is zero/NaN
            min_val = group[variable].min()
            max_val = group[variable].max()
            if min_val == max_val:
                return pd.Series(0, index=group.index)
            return (group[variable] - min_val) / (max_val - min_val) * 2 - 1
        
        return (group[variable] - mean_val) / std_val

    # Calculate z-scores for each participant
    df["z_score"] = df.groupby(participant_col).apply(
        compute_participant_zscores
    ).reset_index(level=0, drop=True)

    # Remove outliers
    df = df.loc[df["z_score"].abs() <= z_threshold].copy()

    # Average z-scores across participants for each coordinate
    grouped = (
        df.groupby(["longitude_corrected", "latitude_corrected"], dropna=False)["z_score"]
        .mean()
        .reset_index()
    )

    # grouped['z_score'] = grouped['z_score'].rolling(window=15, win_type='gaussian', center=True).mean(std=3).fillna(method='bfill').fillna(method='ffill')


    if grouped.empty:
        print("No valid data after grouping. Nothing to plot.")
        return

    # Create GeoDataFrame
    grouped["geometry"] = [
        Point(xy) for xy in zip(grouped["longitude_corrected"], grouped["latitude_corrected"])
    ]
    gdf = gpd.GeoDataFrame(grouped, geometry="geometry", crs="EPSG:4326")

    # Create intervals for plotting
    vlim = max(abs(gdf["z_score"].min()), abs(gdf["z_score"].max()))
    bin_edges = np.linspace(-vlim, vlim, 6)  # 5 bins
    
    # Ensure bin edges are unique
    bin_edges = np.unique(bin_edges)
    if len(bin_edges) < 2:
        bin_edges = np.array([bin_edges[0] - 0.5, bin_edges[0] + 0.5])

    # Create intervals
    try:
        gdf["z_interval"] = pd.cut(
            gdf["z_score"],
            bins=bin_edges,
            include_lowest=True,
            labels=[f"{bin_edges[i]:.1f} to {bin_edges[i+1]:.1f}" 
                   for i in range(len(bin_edges)-1)]
        )
    except ValueError as e:
        print(f"Warning: Fallback to quartile binning due to: {e}")
        gdf["z_interval"] = pd.qcut(
            gdf["z_score"],
            q=4,
            duplicates='drop',
            labels=['Q1', 'Q2', 'Q3', 'Q4']
        )

    # Convert to Web Mercator
    try:
        gdf_web = gdf.to_crs(epsg=3857)
    except Exception as e:
        print(f"Could not convert to EPSG:3857 (Web Mercator): {e}")
        gdf_web = None

    # Create coordinate transformer
    transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

    def transform_x(x, _):
        """Transform Web Mercator x coordinate to longitude"""
        lon, _ = transformer.transform(x, 0)
        return f"{lon:.3f}°"

    def transform_y(y, _):
        """Transform Web Mercator y coordinate to latitude"""
        _, lat = transformer.transform(0, y)
        return f"{lat:.3f}°"

    # Plot
    fig, ax = plt.subplots(figsize=(15, 10))

    if gdf_web is not None:
        gdf_web.plot(
            ax=ax,
            column="z_interval",
            cmap="coolwarm",
            alpha=0.8,
            legend=True,
            markersize=40,
        )
        try:
            ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
        except Exception as e:
            print(f"Could not add basemap with contextily: {e}")
        
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(transform_x))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(transform_y))
    else:
        gdf.plot(
            ax=ax,
            column="z_interval",
            cmap="coolwarm",
            alpha=0.8,
            legend=True,
            markersize=40,
        )
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}°"))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.1f}°"))
        
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)
    ax.set_title(title, fontsize=14)

    # Handle plot saving
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
    
    plt.show()


# Run
plot_participant_averaged_zmap(
    df=data_within_path,
    variable='theta-beta ratio',
    participant_col='participant_id',
    title='Average Z-Scores Across Participants - theta-beta ratio',
    output_path= os.path.join(output_dir,"Maat.png")
)

##### Loop across sessions

In [ ]:
def plot_z_normalized_map_with_intervals2(df, variable, z_threshold=3, title="Z-Normalized Map (With Intervals)", output_path=None, rolling_window=30):
    """
    Plot z-normalized values of a variable on a map with intervals.
    Includes improved binning logic and handling of edge cases.
    
    Args:
        df (pd.DataFrame): DataFrame containing 'geometry' (WKT or shapely objects) and the variable.
        variable (str): The column to z-normalize and plot.
        z_threshold (float): Threshold for removing outliers by absolute z-score. Default is 3.
        title (str): Title for the plot.
        output_path (str, optional): Path to save the plot. If None, plot is only displayed.
        rolling_window (int, optional): Number of consecutive samples to average before computing z-score. Default is 15.

    Returns:
        None: Displays and optionally saves the plot.
    """
    import pandas as pd
    import numpy as np
    import geopandas as gpd
    import shapely.wkt
    import contextily as ctx
    import matplotlib.pyplot as plt
    from shapely.geometry import Point
    import matplotlib.ticker as mticker
    from pyproj import Transformer
    import warnings
    warnings.filterwarnings('ignore', category=UserWarning)

    # Ensure geometry objects
    def ensure_geom(x):
        if isinstance(x, str):
            return shapely.wkt.loads(x)
        return x

    df = df.copy()
    if "geometry" not in df.columns:
        print("No 'geometry' column found in the DataFrame.")
        return

    df["geometry"] = df["geometry"].apply(ensure_geom)
    df.dropna(subset=["geometry"], inplace=True)

    if df.empty:
        print("No valid geometry after dropping NaNs.")
        return

    # Extract coordinates
    df["longitude"] = df["geometry"].apply(lambda g: g.x if g is not None else np.nan)
    df["latitude"] = df["geometry"].apply(lambda g: g.y if g is not None else np.nan)

    # Apply rolling mean to the variable before z-score computation
    if variable in df.columns:
        df[variable] = df[variable].rolling(window=rolling_window, min_periods=1).mean()

    # Group by coordinates
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    skip_cols = {"longitude", "latitude"}
    numeric_cols = [c for c in numeric_cols if c not in skip_cols]

    if not numeric_cols:
        print("No numeric columns found to average.")
        return

    grouped = (
        df.groupby(["longitude", "latitude"], dropna=False)[numeric_cols]
        .mean(numeric_only=True)
        .reset_index()
    )

    if grouped.empty:
        print("Grouping returned an empty DataFrame. Nothing to plot.")
        return

    # Create GeoDataFrame
    grouped["geometry"] = [
        Point(xy) for xy in zip(grouped["longitude"], grouped["latitude"])
    ]
    gdf = gpd.GeoDataFrame(grouped, geometry="geometry", crs="EPSG:4326")

    if variable not in gdf.columns:
        print(f"Variable '{variable}' not found after grouping. Nothing to plot.")
        return

    if gdf[variable].dropna().empty:
        print(f"All values for '{variable}' are NaN or nonexistent. Nothing to plot.")
        return

    # Compute z-scores with robust handling of edge cases
    mean_val = gdf[variable].mean()
    std_val = gdf[variable].std()

    if std_val == 0 or np.isnan(std_val):
        print(f"Warning: Std is zero/NaN for '{variable}'; using min-max scaling instead.")
        min_val = gdf[variable].min()
        max_val = gdf[variable].max()
        if min_val == max_val:
            gdf["z_score"] = 0  # All values are the same
        else:
            gdf["z_score"] = (gdf[variable] - min_val) / (max_val - min_val) * 2 - 1
    else:
        gdf["z_score"] = (gdf[variable] - mean_val) / std_val

    # Remove outliers
    gdf = gdf.loc[gdf["z_score"].abs() <= z_threshold].copy()
    if gdf.empty:
        print(f"All data removed by outlier cutoff (|z|>{z_threshold}). Nothing to plot.")
        return

    # Improved binning logic
    data_range = gdf["z_score"].max() - gdf["z_score"].min()
    n_unique_values = len(gdf["z_score"].unique())
    
    if n_unique_values <= 5:
        # If we have 5 or fewer unique values, use those as the bins
        bin_edges = sorted(gdf["z_score"].unique())
        if len(bin_edges) == 1:
            # Handle case with only one unique value
            bin_edges = [bin_edges[0] - 0.5, bin_edges[0] + 0.5]
    else:
        # Create 5 bins with equal width
        vlim = max(abs(gdf["z_score"].min()), abs(gdf["z_score"].max()))
        bin_edges = np.linspace(-vlim, vlim, 6)  # 5 bins plus 1 edge
        
        # Ensure bin edges are unique
        bin_edges = np.unique(bin_edges)
        if len(bin_edges) < 2:
            # If we still don't have enough unique edges, create artificial spread
            bin_edges = np.array([bin_edges[0] - 0.5, bin_edges[0] + 0.5])

    # Create intervals
    try:
        gdf["z_interval"] = pd.cut(
            gdf["z_score"],
            bins=bin_edges,
            include_lowest=True,
            labels=[f"{bin_edges[i]:.1f} to {bin_edges[i+1]:.1f}" 
                   for i in range(len(bin_edges)-1)]
        )
    except ValueError as e:
        print(f"Warning: Fallback to quartile binning due to: {e}")
        gdf["z_interval"] = pd.qcut(
            gdf["z_score"],
            q=4,
            duplicates='drop',
            labels=['Q1', 'Q2', 'Q3', 'Q4']
        )

    # Convert to Web Mercator
    try:
        gdf_web = gdf.to_crs(epsg=3857)
    except Exception as e:
        print(f"Could not convert to EPSG:3857 (Web Mercator): {e}")
        gdf_web = None

    # Create coordinate transformer
    transformer = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

    def transform_x(x, _):
        """Transform Web Mercator x coordinate to longitude"""
        lon, _ = transformer.transform(x, 0)
        return f"{lon:.3f}°"

    def transform_y(y, _):
        """Transform Web Mercator y coordinate to latitude"""
        _, lat = transformer.transform(0, y)
        return f"{lat:.3f}°"

    # Plot
    fig, ax = plt.subplots(figsize=(15, 10))

    if gdf_web is not None:
        gdf_web.plot(
            ax=ax,
            column="z_interval",
            cmap="coolwarm",
            alpha=0.8,
            legend=True,
            markersize=40,
        )
        try:
            ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
        except Exception as e:
            print(f"Could not add basemap with contextily: {e}")
        
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(transform_x))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(transform_y))
    else:
        gdf.plot(
            ax=ax,
            column="z_interval",
            cmap="coolwarm",
            alpha=0.8,
            legend=True,
            markersize=40,
        )
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}°"))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.1f}°"))
        
    ax.set_xlabel("Longitude", fontsize=12)
    ax.set_ylabel("Latitude", fontsize=12)
    ax.set_title(title, fontsize=14)

    # Handle plot saving
    if output_path:
        plt.savefig(os.path.join(output_path, f"{variable}_interval.png"), dpi=300, bbox_inches='tight')
    
    # plt.tight_layout()
    # plt.close()

eeg_metrics = [
    # EEG variables
    "delta","theta","alpha","beta","gamma","frontal alpha","frontal midline theta",
    "theta-beta ratio","frontal alpha asymmetry","frontal theta",
]
main_sessions = ['Gulbenkian', 'Parque', 'Belem', 'Graca', 'Baixa', 'Lapa']

for session in main_sessions:
    for eeg_metric in eeg_metrics:
        # Import
        input_dir  = os.path.join(results, 'fulldata_analysis','within_path', session)
        data_within_path = pd.read_csv(os.path.join(input_dir, f'{session}.csv'))
        # Plot
        plot_z_normalized_map_with_intervals2(data_within_path, variable=eeg_metric, title=f"Z-Normalized {eeg_metric} Map ({session})", output_path=None)
        # Save
        plt.savefig(os.path.join(input_dir, f'{session}_{eeg_metric}_interval.png'))

#### Between-path

In [ ]:
results_dir = os.path.join(results, 'fulldata')
design = 'between_path'
ses = ['Baixa', 'Graca','Parque','Lapa','Gulbenkian','Belem', 'Maat']
data = do_analysis_design(results_dir, design, session=ses)
# Export
results_dir = os.path.join(results, 'fulldata')
output_dir  = os.path.join(results, 'fulldata_analysis',design)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
data.to_csv(os.path.join(output_dir, f'main_paths.csv'), index=False)

In [ ]:
# Plot data onto map
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

def plot_paths_on_map(combined_df, vars_to_plot, output_dir):
    """
    Create subplots for each path in 'session_path',
    coloring points by a chosen variable (one figure per variable).

    combined_df: pd.DataFrame with columns:
        - longitude, latitude (floats)
        - session_path (string)
        - numeric columns to be plotted
    vars_to_plot: list of string column names to visualize
    output_dir: folder to save the generated figures
    """

    # 1) Convert to GeoDataFrame for easy plotting
    #    If you don't have geopandas, skip this & plot with plain matplotlib scatter
    gdf = gpd.GeoDataFrame(
        combined_df.copy(),
        geometry=gpd.points_from_xy(combined_df['longitude'], combined_df['latitude']),
        crs="EPSG:4326"  # typical lat/lon
    )

    # 2) Identify unique paths
    paths = gdf["session_path"].unique()
    paths = np.sort(paths)  # for consistent ordering

    # 3) For each variable in vars_to_plot => single figure => subplots for each path
    for var in vars_to_plot:
        # a) create figure with one row per path, or do 1 x len(paths)
        fig, axes = plt.subplots(
            1, len(paths),
            figsize=(5 * len(paths), 6),
            subplot_kw={'aspect': 'equal'}  # maintain aspect ratio
        )
        # If there's only one path, axes won't be an array
        if len(paths) == 1:
            axes = [axes]

        for i, path_ in enumerate(paths):
            ax = axes[i]
            # Filter data for this path
            sub_gdf = gdf.loc[gdf["session_path"] == path_]

            # b) plot
            # If the var has NaN, you might want to skip or fill them
            sub_gdf = sub_gdf.dropna(subset=[var])

            # Plot points, color by var
            # we can let GeoPandas handle the color mapping via 'column=var'
            # legend=True => colorbar
            # markersize => for better visibility
            if not sub_gdf.empty:
                sub_gdf.plot(
                    column=var,
                    cmap='viridis',
                    legend=True,
                    markersize=30,
                    alpha=0.8,
                    ax=ax
                )
            else:
                ax.text(0.5, 0.5, "No data", ha='center', va='center', transform=ax.transAxes)
            
            # c) titles, etc.
            ax.set_title(f"{path_}: {var}", fontsize=12)
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")

        plt.suptitle(f"Map of {var} by Path", fontsize=15, y=1.02)
        plt.tight_layout()
        # d) Save figure
        out_fname = os.path.join(output_dir, f"{var}_map_all_paths.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)


numeric_variables = ["frontal alpha"]  # adapt to your data

# Demo: read a CSV
# combined_df = pd.read_csv(r"path\to\your\between_path_result.csv")

# Now call the function
plot_paths_on_map(data, numeric_variables, output_dir)

print("Plots generated for each variable, subplots per path.")

# TESTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import shapely.wkt
import contextily as ctx
from shapely.geometry import Point

def add_basemap(ax, zoom):
    xmin, xmax, ymin, ymax = ax.axis()
    basemap, extent = ctx.bounds2img(xmin, ymin, xmax, ymax, zoom=zoom)
    ax.imshow(basemap, extent=extent, interpolation='bilinear')
    # restore original x/y limits
    ax.axis((xmin, xmax, ymin, ymax))

def plot_z_normalized_map_geometry(df, variable, title="Z-Normalized Map"):
    """
    Plot z-normalized values of a specific variable on a map, 
    using a 'geometry' column (in WKT or geometry objects).
    
    1) Convert geometry from WKT if needed
    2) Compute z-score
    3) Reject outliers with abs(z-score) > 5
    4) Reproject to EPSG:3857
    5) Plot with contextily basemap
    """

    if variable not in df.columns:
        print(f"Variable '{variable}' not in DataFrame columns.")
        return
    if "geometry" not in df.columns:
        print("Column 'geometry' not found in DataFrame.")
        return

    # Make a copy to avoid mutating original
    df = df.copy()
    
    # 1) Convert geometry strings to shapely geometries if needed
    def ensure_geom(x):
        if isinstance(x, str):
            return shapely.wkt.loads(x)
        return x

    df["geometry"] = df["geometry"].apply(ensure_geom)

    # Convert to GeoDataFrame with lat/lon (EPSG:4326)
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")
    # Drop rows missing variable or geometry
    gdf = gdf.dropna(subset=[variable, "geometry"])
    if gdf.empty:
        print("No valid rows after dropping NaNs in geometry/variable.")
        return

    # 2) Compute z-score
    mean_val = gdf[variable].mean()
    std_val  = gdf[variable].std()
    if std_val == 0 or np.isnan(std_val):
        print("Std is zero/NaN, using raw values for variable.")
        gdf["z_score"] = gdf[variable]
    else:
        gdf["z_score"] = (gdf[variable] - mean_val) / std_val

    # 3) Reject outliers with abs(z-score) > 5
    gdf = gdf.loc[gdf["z_score"].abs() <= 5].copy()
    if gdf.empty:
        print("All points were outliers (|z|>5). Nothing to plot.")
        return

    # 4) Reproject to EPSG:3857 for contextily
    gdf_3857 = gdf.to_crs(epsg=3857)

    # 5) Plot
    ax = gdf_3857.plot(figsize=(10, 10), alpha=0.5, edgecolor='k')

    # Plot the points color-coded by z_score
    gdf_3857.plot(
        ax=ax,
        column="z_score",
        cmap="coolwarm",
        legend=True,
        alpha=0.7,
        markersize=30
    )
    
    # Adjust the extent to data
    ax.set_xlim(gdf_3857.total_bounds[0], gdf_3857.total_bounds[2])
    ax.set_ylim(gdf_3857.total_bounds[1], gdf_3857.total_bounds[3])

    # Labels and title
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Longitude (web-mercator)", fontsize=12)
    ax.set_ylabel("Latitude (web-mercator)", fontsize=12)

    add_basemap(ax, zoom=10)
    ax.set_axis_off()
    # plt.tight_layout()
    # plt.show()


# Example usage:
# sub_data = pd.read_csv("some_file.csv")  # containing 'variable' and 'geometry'
plot_z_normalized_map_geometry(sub_data, variable="frontal alpha", 
                               title="Z-Normalized frontal alpha Map")


